# Symmetry TTA + v3×v2 cross-soup

Blend: **`(1-α)·v2_step2000 + α·v3_step400`**, best **α≈0.10**.

Submit: **symmetry TTA** + **pair_text v1** inline.

См. [README.md](./README.md)

In [ ]:
import json
import sys
from pathlib import Path

NB = Path.cwd()
sys.path.insert(0, str(NB.parent / "lib"))
from nb_setup import setup, run

PATHS = setup()
REPO = PATHS["repo"]
SCRIPTS = PATHS["scripts"]
OUT = NB / "output" / "v3_soup_tta_run"
OUT.mkdir(parents=True, exist_ok=True)

V2 = PATHS["v2_ckpt_root"]
CKPT_A = PATHS["v3_train"] / "checkpoints/step_00400.pt"
CKPT_B = V2 / "checkpoints/step_02000.pt"
REFERENCE = PATHS["runs_tta"]
SUBMIT_SRC = REPO / "final_4_models/submit/matching-bge-human-ft"

print("REPO:", REPO)
print("stage_a init:", PATHS["stage_a_init"], PATHS["stage_a_init"].exists())
print("v3 ckpt:", CKPT_A, CKPT_A.exists())
print("v2 ckpt root:", V2)
print("v2 ckpt:", CKPT_B, CKPT_B.exists())

## 1. Verify pair_text v1

In [ ]:
from verify_pair_text import main as verify_main, compare_v1_vs_v2
assert verify_main() == 0

# submit template uses same v1 logic (inline in utils.py)
utils_src = (SUBMIT_SRC / "src/utils.py").read_text()
assert "symmetry TTA" in utils_src or "enc_rev" in utils_src
assert "[:520]" in utils_src, "submit must use attr cap 520 (v1)"
print("submit template: v1 + symmetry TTA OK")
print(json.dumps(compare_v1_vs_v2(), ensure_ascii=False, indent=2)[:500])

## 2. (Optional) Stage-B v3 train

Output: `user_bge_stageb_final_run/` (gray distill, EMA).

In [ ]:
RUN_V3 = False
if RUN_V3:
    run(f"cd {SCRIPTS} && python train_bge_stageb_v3.py --precompute-teacher")
    run(f"cd {SCRIPTS} && CUDA_VISIBLE_DEVICES=0,1 torchrun --nproc_per_node=2 train_bge_stageb_v3.py")

## 3. Cross-soup blend

In [ ]:
RUN_BLEND = True
if RUN_BLEND:
    cmd = (
        f"cd {REPO} && PYTHONPATH={SCRIPTS} python {SCRIPTS}/blend_checkpoint_soup.py "
        f"--ckpt-a {CKPT_A} --ckpt-b {CKPT_B} "
        f"--out-dir {OUT} --alphas 0.10,0.15,0.20,0.25 --skip-submit --gpu 0"
    )
    assert run(cmd) == 0

if (OUT / "metrics.json").exists():
    print(json.dumps(json.loads((OUT / "metrics.json").read_text())["best_metrics"], indent=2))

## 4. Reference offline metrics (04_v3_soup_tta)

In [ ]:
manifest = json.loads((REPO / "final_4_models/manifest.json").read_text())
m04 = next(m for m in manifest["models"] if m["id"] == "04_v3_soup_tta")
print("offline:", m04["offline"])
print("symmetry_tta:", m04["symmetry_tta"])
print("submit sha:", m04["submit_sha256"])

## 5. Build submit (symmetry TTA template)

In [ ]:
run_dir = OUT if (OUT / "export_fp16").exists() else REFERENCE
cmd = f"python {SCRIPTS}/build_bge_human_submit.py --run-dir {run_dir} --full-only"
assert run(cmd) == 0
zip_path = run_dir / "matching-bge-human-ft-submit.zip"
print("zip:", zip_path)

## 6. TTA sanity (optional, needs GPU + sample data)

Symmetry TTA усредняет `p(a,b)` и `p(b,a)` — см. `_score_pairs` в submit utils.

In [ ]:
import inspect
sys.path.insert(0, str(SUBMIT_SRC.parent.parent))
# show TTA scoring signature from frozen submit
src = (SUBMIT_SRC / "src/utils.py").read_text().split("def _score_pairs")[1][:600]
print(src[:500])